In [15]:
import pyautogui as pyg
pyg.mouseInfo()


In [2]:
import pyautogui as pyg
pyg.sleep(2)
basex=327
basey=384
gap=40

In [ ]:
def main():

    print("寶石方塊自動消除啟動")
    print("請先進入遊戲並確認棋盤位置")
    print("按 Q 可以停止")
    print("移動滑鼠到左上角也可以緊急停止")

    start_time = time.time()
    moves = 0
    no_move_count = 0

    while True:

        # 手動停止
        if keyboard.is_pressed("q"):
            print("使用者停止")
            break

        # 時間停止
        if time.time() - start_time >= MAX_RUNTIME:
            print("執行時間結束")
            break

        # 讀取真實螢幕
        img = capture_board()

        board = read_board(img)

        # 尋找最佳消除
        move, score = find_best_move(board)

        if move is None:

            no_move_count += 1

            print("目前沒有找到可消除組合")

            if no_move_count >= 8:
                print("連續無法找到有效移動，停止")
                break

            time.sleep(0.4)
            continue

        no_move_count = 0

        print(
            f"第 {moves+1} 次交換 | "
            f"位置：{move} | "
            f"消除分數：{score}"
        )

        # 執行交換
        perform_move(move)

        moves += 1

        # 等待寶石消除及掉落
        time.sleep(WAIT_TIME)

    print("====================")
    print("自動消除結束")
    print("總交換次數：", moves)


if __name__ == "__main__":
    main()

In [29]:
import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 308,375 
RIGHT, BOTTOM = 629, 693
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 62 # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 1 # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8 # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 2.0 # 等棋盤穩定的最長時間（秒）
VERBOSE = True # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55: # 太暗 → 空格
        return "empty"
    if s < 45: # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print(" " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f" ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path) # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


import ctypes
import sys
import time

import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 495, 371
RIGHT, BOTTOM = 820, 691
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)


def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)


# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 62 # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 10 # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8 # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 2.0 # 等棋盤穩定的最長時間（秒）
VERBOSE = True # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}

SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}


# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)


def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)


# =========================
# 顏色辨識
# =========================
def identify(patch):
    """把一小塊畫面（BGR）轉成寶石顏色名稱。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    h, s, v = (int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

    if v < 55: # 太暗 → 空格
        return "empty"
    if s < 45: # 幾乎沒有飽和度 → 白色 / 特殊寶石
        return "white"
    if h < 8 or h >= 170:
        return "red"
    if h < 22:
        return "orange"
    if h < 35:
        return "yellow"
    if h < 85:
        return "green"
    if h < 115:
        return "blue"
    return "purple"


def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - 4:y + 5, x - 4:x + 5]
            row.append(identify(patch))
        board.append(row)
    return board


def print_board(board):
    for row in board:
        print(" " + " ".join(SYMBOL.get(c, "?") for c in row))


# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()

    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end

    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end

    return found


# 舊名稱，方便沿用
matches = find_matches


def best_move(board):
    """回傳 (a, b, score)；找不到就回傳 None。"""
    best = None
    best_score = 0

    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue

                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE:
                    continue
                if color == other:
                    continue

                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color

                found = find_matches(test)

                # 交換的位置必須真的被消掉，才算是有效步
                if (r, c) not in found and (r2, c2) not in found:
                    continue

                score = len(found)
                if score > best_score:
                    best_score = score
                    best = ((r, c), (r2, c2), score)

    return best


def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7


def signature(board):
    return tuple(tuple(row) for row in board)


# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")


def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None

    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)

    return board


# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)

    print("\n各格中心 HSV（h, s, v）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            h, s, v = (int(n) for n in np.median(
                hsv[y - 4:y + 5, x - 4:x + 5].reshape(-1, 3), axis=0))
            print(f" ({r},{c}) -> {board[r][c]:7s} h={h:3d} s={s:3d} v={v:3d}")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", img)
    if ok:
        buf.tofile(path) # 用 tofile 才支援中文路徑
        print(f"\n截圖已存成 {path}")


# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]

    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"

    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"

    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"

    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"

    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"

    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)

    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")


# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}~{RIGHT}, y {TOP}~{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)

    start = time.time()
    moves = 0
    still = 0
    last_board = None

    while True:
        if stop_pressed():
            print("手動停止")
            break

        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())

        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break

        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]

        move = best_move(board)

        if move is None:
            still += 1
            print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")
            if still >= STILL_LIMIT:
                print("連續找不到步可走，判定遊戲結束")
                break
            time.sleep(0.4)
            continue

        still = 0
        a, b, score = move
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()

    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "selftest":
        selftest()
    else:
        main()


寶石方塊 自動消除
棋盤範圍：x 308~629, y 375~693
1 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
 O P B G R P G Y
 O P G W W R G W
 W R Y P B P P O
 G P W P G P Y O
 B P G O W Y P W
 O G R Y R Y R O
 R G R O G O O R
 B P O R O P P B
[  0.0s] 交換 (3, 5) <-> (3, 6)，預計消除 6 顆
 O P B G R O G Y
 O P G W W O P W
 W R Y P B B O O
 G P W P W P G O
 G P G O W R G W
 O G R Y R P R O
 R G R O G O O R
 B P O R O P P B
[  1.7s] 交換 (6, 4) <-> (7, 4)，預計消除 4 顆
 O P B G R R B Y
 O P G G W O G W
 W R Y W B O P O
 G P W P W B O O
 O P G P W P G W
 O G R O R R G O
 R G R Y O P R R
 B P O R G P P B
[  3.1s] 交換 (2, 3) <-> (2, 4)，預計消除 4 顆
 O P B G O R B Y
 O P G G B O G W
 W R Y B P O P O
 G P W P R B O O
 B P G P W P G W
 O G R O R R G O
 R G R Y O P R R
 B P O R G P P B
[  4.8s] 交換 (0, 4) <-> (0, 5)，預計消除 3 顆
 O P B G R . B Y
 O P G G B Y G W
 W R Y B P . P O
 G P W P R B O O
 G P G P W P G W
 O G R O R R G O
 R G R Y O P R R
 B P O R G P P B
[  5.9s] 交換 (1, 1) <-> (2, 1)，預計消除 3 顆
 O P B G R B B Y
 O R G G B O G W
 W . Y B P O P O
 G .